In [2]:
import pdfplumber
import pandas as pd
import re
from pathlib import Path

# =========================
# FILES
# =========================

PDF_FILE = Path("national post code directory.pdf")
DERIVATION_CSV = Path("Pakistan Post Code  Derivation.csv")

OUTPUT_FOLDER = Path("final_output1")
OUTPUT_FOLDER.mkdir(exist_ok=True)

GPO_CSV = OUTPUT_FOLDER / "gpo_filtered.csv"
FINAL_CSV = OUTPUT_FOLDER / "new_Pakistan_Post_Code_Derivation.csv"


# =========================
# CLEANING FUNCTIONS
# =========================

def clean_text(v):
    if pd.isna(v):
        return ""
    return str(v).replace("\xa0", " ").strip()


def normalize(v):
    v = clean_text(v).upper()

    v = re.sub(r"\bGPO\b", "", v)
    v = re.sub(r"\bDISTRICT\b", "", v)

    v = re.sub(r"[.,-]", " ", v)
    v = re.sub(r"\s+", " ", v)

    return v.strip()


def is_missing(v):
    return v is None or str(v).strip() == ""


# =========================
# STEP 1 - EXTRACT GPO
# =========================

def extract_gpo():
    data = []

    with pdfplumber.open(PDF_FILE) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()

            for table in tables:
                for row in table:

                    if not row:
                        continue

                    row = [clean_text(x) for x in row]

                    if len(row) < 2:
                        continue

                    name = row[0]

                    if str(name).upper().endswith("GPO"):

                        clean_name = re.sub(
                            r"\s+GPO$",
                            "",
                            name,
                            flags=re.IGNORECASE
                        ).strip()

                        text = " ".join(row)

                        codes = re.findall(r"\b\d{5}\b", text)

                        post_code = codes[0] if codes else ""

                        data.append([
                            clean_name,
                            normalize(clean_name),
                            post_code
                        ])

    df = pd.DataFrame(
        data,
        columns=["name", "key", "post_code"]
    )

    df.to_csv(GPO_CSV, index=False)

    return df


# =========================
# STEP 2 - LOAD FILE
# =========================

def load_derivation():

    df = pd.read_csv(
        DERIVATION_CSV,
        dtype=str
    ).fillna("")

    post_col = None
    city_col = None
    district_col = None

    for c in df.columns:

        lc = c.lower()

        if "post" in lc:
            post_col = c

        if "city" in lc:
            city_col = c

        if "district" in lc:
            district_col = c

    return df, post_col, city_col, district_col


# =========================
# STEP 3 - FILL POST CODES
# =========================

def fill_postcodes(base_df, gpo_df, post_col, city_col, district_col):

    mapping = dict(
        zip(gpo_df["key"], gpo_df["post_code"])
    )

    final = []
    match_used = []

    filled = 0

    for _, row in base_df.iterrows():

        current = row.get(post_col, "")
        city = row.get(city_col, "")
        district = row.get(district_col, "")

        # already exists
        if not is_missing(current):

            final.append(current)
            match_used.append("existing")
            continue

        match_key = ""

        if not is_missing(city):
            match_key = normalize(city)

        elif not is_missing(district):
            match_key = normalize(district)

        else:
            final.append("")
            match_used.append("no data")
            continue

        post = mapping.get(match_key, "")

        if post:
            filled += 1
            final.append(post)
            match_used.append(match_key)
        else:
            final.append("")
            match_used.append(match_key)

    base_df[post_col] = final
    base_df["match_key_used"] = match_used

    print("Filled Post Codes:", filled)

    return base_df


# =========================
# MAIN
# =========================

def main():

    print("Extracting GPO...")
    gpo_df = extract_gpo()

    print("Loading derivation file...")
    base_df, post_col, city_col, district_col = load_derivation()

    print("Merging + filling post codes...")
    final_df = fill_postcodes(
        base_df,
        gpo_df,
        post_col,
        city_col,
        district_col
    )

    print("Saving final file...")
    final_df.to_csv(FINAL_CSV, index=False)

    print("\nDONE ✔")
    print("Saved:", FINAL_CSV)


if __name__ == "__main__":
    main()

Extracting GPO...
Loading derivation file...
Merging + filling post codes...
Filled Post Codes: 142613
Saving final file...

DONE ✔
Saved: final_output1\new_Pakistan_Post_Code_Derivation.csv
